# Zadanie 5: programowanie genetyczne i regresja symboliczna

Adam Tokarz, Serhii Zeliuk

EAIIIB ISI Metody i algorytmy optymalizacji grupa laboratoryjna 2 poniedziałek 18:30

Termin realizacji: 12 maja 2025

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zmodyfikuj przykład `pysr_demo.ipynb` tak, aby uczył się funkcji $f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$ której dziedziną jest $\mathbb{R}^6$. Uczenie ma się odbywać w oparciu o 200 wylosowanych z dziedziny próbek (między -5 a 5).
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^").
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


In [1]:
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


# 5.1 Function f [3.0]

$$f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$$

- X (random values  $\mathbb{R}^6$ [-5, 5])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev = 0.5) 

In [2]:
from numpy import sin, cos, exp, log, sqrt
# Dataset

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
# y = 2.5382 * np.cos(X[:, 3]) + X[:, 0] ** 2 - 2
y = 2.2*sin(X[:, 0] + 2*X[:, 1]) - X[:, 5] ** 2 - 3
# Add the y samples with random noise
noise = np.random.normal(loc=0.0, scale=0.5, size=y.shape)
y_noisy = y + noise

In [3]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.2 Experiments [3.0]



## 5.2.1 Model 1

In [4]:
model_1 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.158
3           5.546e+01  1.240e-03  y = x₂ + -11.311
4           5.152e+01  7.370e-02  y = cos(x₅) + -10.987
5           6.759e+00  2.031e+00  y = (x₅ * x₅) * -1.1931
7           2.205e+00  5.600e-01  y = ((x₅ * x₅) * -0.98155) + -3.1631
9           2.197e+00  1.884e-03  y = (((x₅ + 0.033222) * -0.9792) * x₅) + -3.1778
11          2.173e+00  5.516e-03  y = (x₀ * 0.059193) + (((x₅ * x₅) * -0.98035) + -3.1557)
12          2.157e+00  7.206e-03  y = (cos(x₀ * -0.25229) + (x₅ * (x₅ * -0.98016))) + -3.900...
                                      3
13          2.051e+00  5.052e-02  y = ((x₅ * (-0.98296 * x₅)) + cos(x₁ + cos(x₁))) + -3.0716
15          1.749e+00  7.975e-02  y = (cos(cos(x₁ + -0.70005) * x₀) + -3.2906) + (x₅ * (x₅ *...
                                       -0.98633))
──────────────────────────

PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                          -11.15813   
	1        0.001240                                    x2 + -11.310952   
	2        0.073704                               cos(x5) + -10.987488   
	3        2.031154                             (x5 * x5) * -1.1931036   
	4  >>>>  0.560019              ((x5 * x5) * -0.9815461) + -3.1631167   
	5        0.001884  (((x5 + 0.033221856) * -0.97919905) * x5) + -3...   
	6        0.005516  (x0 * 0.059192885) + (((x5 * x5) * -0.98035324...   
	7        0.007206  (cos(x0 * -0.2522873) + (x5 * (x5 * -0.9801627...   
	8        0.050520  ((x5 * (-0.9829572 * x5)) + cos(x1 + cos(x1)))...   
	9        0.079754  (cos(cos(x1 + -0.7000452) * x0) + -3.2906103) ...   
	
	        loss  complexity  
	0  55.599790           1  
	1  55.462025           3  
	2  51.521240           4  
	3   6.758767           5  
	4   2.205166           7  
	5   2.196871           9  
	6   2.172767          11  
	7   2.157165          12  
	8   2.050893          13  
	9   1.748517          15  
]

### Model 1 best + top3 according to score

In [5]:
best_equation_1 = model_1.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x5*x5*(-0.9815461) - 3.1631167
Index 3, score=2.031154
Equation: (x5 * x5) * -1.1931036

Index 4, score=0.560019
Equation: ((x5 * x5) * -0.9815461) + -3.1631167

Index 9, score=0.079754
Equation: (cos(cos(x1 + -0.7000452) * x0) + -3.2906103) + (x5 * (x5 * -0.98633015))



## 5.2.2 Model 2

Added constraint to operator "^" - safe_pow

In [6]:
%%julia
function safe_pow(x::T, y::T) where T
    if(x < 0 && floor(y) != y)
        return T(NaN)
    else
        return T(x^y)
    end
end

safe_pow (generic function with 1 method)

In [7]:
def safe_pow(x, y):
    if x < 0 and np.floor(y) != y:
        return np.nan
    return np.power(x, y)

model_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.159
3           5.537e+01  2.051e-03  y = -11.021 - x₅
4           5.152e+01  7.208e-02  y = cos(x₅) + -10.987
5           2.224e+00  3.143e+00  y = -3.0128 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = ((x₅ * x₅) * -0.98155) - 3.1631
9           2.194e+00  2.514e-03  y = -2.9965 - ((x₀ * -0.056602) + (x₅ * x₅))
10          2.134e+00  2.777e-02  y = -3.0123 - ((x₅ * x₅) + (sin(x₀) * 0.41035))
11          2.132e+00  1.090e-03  y = -3.1804 - (((x₅ * 0.98022) + (x₄ * -0.033961)) * x₅)
12          2.106e+00  1.219e-02  y = (-3.271 - ((x₅ * x₅) + (sin(x₀) * 0.4434))) * 0.97735
13          2.065e+00  1.940e-02  y = -3.0738 - ((x₅ * x₅) + sin(sin((x₁ + 0.035177) * 2.056...
                                      7)))
16          2.055e+00  1.707e-03  y = -3.1521 - ((((x₀ - 4.3501) * -0.098488) * sin(x₀)) + (.

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.158837   
	1         0.002051                                    -11.020786 - x5   
	2         0.072084                               cos(x5) + -10.987483   
	3         3.142667                              -3.012776 - (x5 * x5)   
	4         0.004262              ((x5 * x5) * -0.98154575) - 3.1631072   
	5         0.002514     -2.9964771 - ((x0 * -0.056602404) + (x5 * x5))   
	6         0.027774  -3.0123422 - ((x5 * x5) + (sin(x0) * 0.41035104))   
	7         0.001090  -3.1803837 - (((x5 * 0.98021626) + (x4 * -0.03...   
	8         0.012194  (-3.2709582 - ((x5 * x5) + (sin(x0) * 0.443398...   
	9         0.019396  -3.07385 - ((x5 * x5) + sin(sin((x1 + 0.035177...   
	10        0.001707  -3.1521459 - ((((x0 - 4.3501344) * -0.09848773...   
	11        0.806121  ((safe_pow(0.18791056, cos((x1 + -1.5706439) +...   
	12        0.672732  ((safe_pow(0.18773219, sin(cos((x1 + (x1 + -1....   
	13  >>>>  0.724485  (safe_pow(0.42306387, cos(x1 + (x0 + (x1 + -1....   
	
	         loss  complexity  
	0   55.599790           1  
	1   55.372204           3  
	2   51.521244           4  
	3    2.224043           5  
	4    2.205166           7  
	5    2.194108           9  
	6    2.134008          10  
	7    2.131682          11  
	8    2.105847          12  
	9    2.065395          13  
	10   2.054845          16  
	11   0.409818          18  
	12   0.209135          19  
	13   0.101342          20  
]

  - outputs\20250511_133152_Rrmxli\hall_of_fame.csv


In [8]:
best_equation_2 = model_2.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: (0.42306387**(cos(x0 + x1 + x1 - 1.5642786) - 0.87726414) + x5*x5 + 0.53179604)*(-0.99682945)
Index 3, score=3.142667
Equation: -3.012776 - (x5 * x5)

Index 11, score=0.806121
Equation: ((safe_pow(0.18791056, cos((x1 + -1.5706439) + (x1 + x0))) + (x5 * x5)) + 1.4434937) * -0.999068

Index 13, score=0.724485
Equation: (safe_pow(0.42306387, cos(x1 + (x0 + (x1 + -1.5642786))) + -0.87726414) + ((x5 * x5) + 0.53179604)) * -0.99682945



## 5.2.3 Model 3

In [9]:
model_3 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.159
3           5.537e+01  2.051e-03  y = -11.021 - x₅
4           5.462e+01  1.364e-02  y = -11.157 - sin(x₀)
5           2.224e+00  3.201e+00  y = -3.0128 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = (x₅ * (x₅ * -0.98155)) + -3.1631
9           2.153e+00  1.189e-02  y = -3.0191 - ((x₅ + (x₄ * -0.03329)) * x₅)
10          2.134e+00  9.020e-03  y = -3.0123 - ((x₅ * x₅) + (sin(x₀) * 0.41076))
11          2.133e+00  4.447e-04  y = -3.0114 - ((sin(sin(x₀)) * 0.47014) + (x₅ * x₅))
12          2.132e+00  6.522e-04  y = -3.1804 - ((x₅ * 0.98021) * (x₅ + sin(x₄ * -0.034776))...
                                      )
13          2.105e+00  1.282e-02  y = (-3.2719 - ((sin(sin(x₀)) * 0.50886) + (x₅ * x₅))) * 0...
                                      .97718
───────────────────────────────────────────────────

PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                         -11.158837   
	1        0.002051                                    -11.020789 - x5   
	2        0.013642                                -11.15744 - sin(x0)   
	3  >>>>  3.201109                              -3.012808 - (x5 * x5)   
	4        0.004262             (x5 * (x5 * -0.98154527)) + -3.1631148   
	5        0.011890      -3.019106 - ((x5 + (x4 * -0.033289727)) * x5)   
	6        0.009021   -3.0123043 - ((x5 * x5) + (sin(x0) * 0.4107571))   
	7        0.000445  -3.011416 - ((sin(sin(x0)) * 0.47013983) + (x5...   
	8        0.000652  -3.1804497 - ((x5 * 0.9802147) * (x5 + sin(x4 ...   
	9        0.012820  (-3.2718947 - ((sin(sin(x0)) * 0.5088563) + (x...   
	
	        loss  complexity  
	0  55.599790           1  
	1  55.372204           3  
	2  54.621964           4  
	3   2.224043           5  
	4   2.205166           7  
	5   2.153344           9  
	6   2.134007          10  
	7   2.133058          11  
	8   2.131667          12  
	9   2.104514          13  
]

  - outputs\20250511_133158_4OjeJC\hall_of_fame.csv


In [10]:
best_equation_3 = model_3.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 3.012808
Index 3, score=3.201109
Equation: -3.012808 - (x5 * x5)

Index 2, score=0.013642
Equation: -11.15744 - sin(x0)

Index 9, score=0.012820
Equation: (-3.2718947 - ((sin(sin(x0)) * 0.5088563) + (x5 * x5))) * 0.97718245



## 5.2.4 Model 1 with noise

In [11]:
model_1_noisy.fit(X, y_noisy)
best_equation_1_noisy = model_1_noisy.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1_noisy.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.093
3           5.560e+01  5.027e-04  y = x₂ + -11.245
4           5.159e+01  7.493e-02  y = cos(x₅) + -10.922
5           6.855e+00  2.018e+00  y = x₅ * (x₅ * -1.188)
7           2.449e+00  5.148e-01  y = (x₅ * (x₅ * -0.97982)) + -3.1117
9           2.434e+00  2.937e-03  y = (x₅ * ((x₅ * -0.97674) + -0.042769)) + -3.1309
11          2.363e+00  1.485e-02  y = (x₅ * ((x₅ * -0.97839) + (x₄ * 0.036651))) + -3.1302
13          2.307e+00  1.201e-02  y = (((x₅ * x₅) * -0.98197) + cos(x₁ + cos(x₁))) + -2.9896
15          2.286e+00  4.573e-03  y = ((x₅ * x₅) * -0.97678) + (cos(cos(x₁ + -0.23599) + x₁)...
                                       + -2.9109)
16          2.197e+00  3.972e-02  y = (((x₅ * x₅) * -0.98254) + cos(cos(x₁) + (x₁ + cos(x₁))...
                                      )) + -3.1188
18          2.

## 5.2.5 Model 2 with noise

In [12]:
model_2_noisy.fit(X, y_noisy)
best_equation_2_noisy = model_2_noisy.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2_noisy.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.094
3           5.528e+01  3.374e-03  y = -10.955 - x₅
4           5.159e+01  6.919e-02  y = cos(x₅) - 10.922
5           2.471e+00  3.039e+00  y = -2.9473 - (x₅ * x₅)
7           2.449e+00  4.586e-03  y = ((x₅ * x₅) * -0.97982) - 3.1117
9           2.434e+00  2.937e-03  y = ((x₅ * (x₅ - -0.043762)) + 3.2057) * -0.97674
11          2.365e+00  1.451e-02  y = (((x₅ - (x₄ * 0.042264)) * x₅) - -3.2077) * -0.97893
12          2.346e+00  8.017e-03  y = ((sin(x₀) * 0.45196) + ((x₅ * x₅) - -3.2248)) * -0.975...
                                      55
13          2.342e+00  1.756e-03  y = ((exp(sin(x₀)) * 0.39918) + ((x₅ * x₅) - -2.7036)) * -...
                                      0.97635
15          2.290e+00  1.110e-02  y = (exp(sin(x₁) * sin(cos(x₁))) + ((x₅ * x₅) - -2.1521)) ...
                           

## 5.2.6 Model 3 with noise

In [13]:
model_3_noisy.fit(X, y_noisy)
best_equation_3_noisy = model_3_noisy.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3_noisy.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.093
3           5.528e+01  3.374e-03  y = -10.955 - x₅
4           5.467e+01  1.109e-02  y = -11.092 - sin(x₀)
5           2.471e+00  3.097e+00  y = -2.9474 - (x₅ * x₅)
7           2.449e+00  4.585e-03  y = (x₅ * (x₅ * -0.97982)) - 3.1117
9           2.409e+00  8.170e-03  y = (x₅ * ((x₄ * 0.018187) - x₅)) + -2.9494
10          2.378e+00  1.268e-02  y = (-2.9468 - (x₅ * x₅)) + (sin(x₀) * -0.41615)
11          2.378e+00  1.895e-05  y = (sin(sin(x₀) * -0.4265) + -2.9467) - (x₅ * x₅)
12          2.346e+00  1.387e-02  y = (sin(x₀) * -0.44089) + ((-3.2244 - (x₅ * x₅)) * 0.9755...
                                      8)
13          2.346e+00  5.591e-05  y = ((sin(sin(x₀) * -0.46559) + -3.2244) - (x₅ * x₅)) * 0....
                                      97553
14          2.333e+00  5.268e-03  y = ((x₅ * (-0.041 

# 5.3 Output into a DF [3.0]

In [14]:
import pandas as pd

data = []

for model_name in ['model_1', 'model_2', 'model_3','model_1_noisy', 'model_2_noisy', 'model_3_noisy']:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,x5*x5*(-0.9815461) - 3.1631167,2.031154
1,model_1,3,(x5 * x5) * -1.1931036,2.031154
2,model_1,4,((x5 * x5) * -0.9815461) + -3.1631167,0.560019
3,model_1,9,(cos(cos(x1 + -0.7000452) * x0) + -3.2906103) ...,0.079754
4,model_2,best,(0.42306387**(cos(x0 + x1 + x1 - 1.5642786) - ...,3.142667
5,model_2,3,-3.012776 - (x5 * x5),3.142667
6,model_2,11,"((safe_pow(0.18791056, cos((x1 + -1.5706439) +...",0.806121
7,model_2,13,"(safe_pow(0.42306387, cos(x1 + (x0 + (x1 + -1....",0.724485
8,model_3,best,-x5*x5 - 3.012808,3.201109
9,model_3,3,-3.012808 - (x5 * x5),3.201109


In [15]:
df.to_csv('results3.csv')

# 5.4 Function f [4.0]

$$f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$$

- X (random values  $\mathbb{R}^6$ [-15, 15])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev1 = 2, std_dev2 = 5) 

In [16]:
from numpy import sin, cos, exp, log, sqrt
# Dataset

np.random.seed(0)
X = np.random.uniform(-15, 15, size=(200, 6))
# y = 2.5382 * np.cos(X[:, 3]) + X[:, 0] ** 2 - 2
y = 2.2*sin(X[:, 0] + 2*X[:, 1]) - X[:, 5] ** 2 - 3
# Add the y samples with random noise
noise1 = np.random.normal(loc=0.0, scale=2, size=y.shape)
y_noisy1 = y + noise1
noise2 = np.random.normal(loc=0.0, scale=5, size=y.shape)
y_noisy2 = y + noise2

In [17]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.5 Experiments [4.0]



## 5.5.1 Model 1

In [18]:
model_1_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)

model_1_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.289
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           6.228e+00  3.273e+00  y = x₅ * (x₅ * -1.0218)
7           2.054e+00  5.546e-01  y = (x₅ * (x₅ * -0.99929)) + -3.0285
9           2.022e+00  7.920e-03  y = (((x₅ + -0.021394) * x₅) * -0.99981) + -2.9996
11          2.020e+00  3.885e-04  y = ((x₅ * ((x₄ * 0.00023595) + -1.0003)) * x₅) + -2.9693
13          1.970e+00  1.265e-02  y = (sin(sin(x₀ * 1.4158)) + -3.0604) + (x₅ * (x₅ * -0.999...
                                      09))
14          6.111e-01  1.170e+00  y = (sin(x₀ + (x₁ + x₁)) + (x₅ * (x₅ * -0.99961))) + -3.01...
                                      55
16          4.364e-11  7.971e+00  y = ((x₅ * x₅) * -1) + ((sin((x₁ + x₁) + x₀) * 2.2) + -3)
18          4.224e-11  1.633e-02  y = ((x₅ * (x₅ * -1)) + -1.9509) + ((sin(x₀ + (x₁ + 

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	   pick      score                                           equation  \
	0         0.000000                                           -76.2889   
	1         0.016651                                     x2 + -76.74523   
	2         3.273026                             x5 * (x5 * -1.0218029)   
	3         0.554636               (x5 * (x5 * -0.9992922)) + -3.028511   
	4         0.007920  (((x5 + -0.021394253) * x5) * -0.9998069) + -2...   
	5         0.000389  ((x5 * ((x4 * 0.00023595469) + -1.0002575)) * ...   
	6         0.012652  (sin(sin(x0 * 1.415828)) + -3.060448) + (x5 * ...   
	7         1.170350  (sin(x0 + (x1 + x1)) + (x5 * (x5 * -0.99961394...   
	8  >>>>  11.681305  ((x5 * x5) * -1.0) + ((sin((x1 + x1) + x0) * 2...   
	9         0.016332  ((x5 * (x5 * -1.0)) + -1.950909) + ((sin(x0 + ...   
	
	           loss  complexity  
	0  4.484837e+03           1  
	1  4.337941e+03           3  
	2  6.228301e+00           5  
	3  2.054086e+00           7  
	4  2.021806e+00           9  
	5  2.020235e+00          11  
	6  1.969758e+00          13  
	7  6.111335e-01          14  
	8  4.363990e-11          16  
	9  4.223750e-11          18  
]

  - outputs\20250511_133208_d5d1pR\hall_of_fame.csv


### Model 1 best + top3 according to score

In [19]:
best_equation_1 = model_1_wide_range.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.200001 - 2.9999998
Index 8, score=11.681305
Equation: ((x5 * x5) * -1.0) + ((sin((x1 + x1) + x0) * 2.200001) + -2.9999998)

Index 2, score=3.273026
Equation: x5 * (x5 * -1.0218029)

Index 7, score=1.170350
Equation: (sin(x0 + (x1 + x1)) + (x5 * (x5 * -0.99961394))) + -3.0155451



## 5.5.2 Model 2

In [20]:

model_2_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=30,
    **default_pysr_params,
)

model_2_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.288
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           2.056e+00  3.827e+00  y = -2.9764 - (x₅ * x₅)
7           2.022e+00  8.427e-03  y = ((0.021677 - x₅) * x₅) + -2.9856
9           2.022e+00  3.997e-05  y = (x₅ * ((0.021377 - x₅) * 0.99981)) + -2.9997
10          1.987e+00  1.745e-02  y = sin(-6.1181 * x₅) - ((x₅ * x₅) - -2.9333)
11          1.950e+00  1.867e-02  y = sin(sin(x₅ * -6.1181)) - ((x₅ * x₅) - -2.9333)
12          1.835e+00  6.070e-02  y = (cos(cos(cos(x₀) + x₁)) - (x₅ * x₅)) - 3.7409
16          1.831e+00  6.353e-04  y = (cos(cos(cos((x₅ * 0.020248) + x₀) + x₁)) - (x₅ * x₅))...
                                       - 3.7391
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_133210_GnbNXP\hall_of_fame.csv


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                         -76.287575   
	1        0.016651                                     x2 + -76.74515   
	2  >>>>  3.827115                             -2.9764156 - (x5 * x5)   
	3        0.008427              ((0.02167654 - x5) * x5) + -2.9855733   
	4        0.000040  (x5 * ((0.021376822 - x5) * 0.9998061)) + -2.9...   
	5        0.017449     sin(-6.118086 * x5) - ((x5 * x5) - -2.9332786)   
	6        0.018667  sin(sin(x5 * -6.118086)) - ((x5 * x5) - -2.933...   
	7        0.060703   (cos(cos(cos(x0) + x1)) - (x5 * x5)) - 3.7408514   
	8        0.000635  (cos(cos(cos((x5 * 0.020248404) + x0) + x1)) -...   
	
	          loss  complexity  
	0  4484.837400           1  
	1  4337.941400           3  
	2     2.056337           5  
	3     2.021969           7  
	4     2.021807           9  
	5     1.986834          10  
	6     1.950090          11  
	7     1.835235          12  
	8     1.830577          16  
]

In [21]:
best_equation_2 = model_2_wide_range.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 2.9764156
Index 2, score=3.827115
Equation: -2.9764156 - (x5 * x5)

Index 7, score=0.060703
Equation: (cos(cos(cos(x0) + x1)) - (x5 * x5)) - 3.7408514

Index 6, score=0.018667
Equation: sin(sin(x5 * -6.118086)) - ((x5 * x5) - -2.9332786)



## 5.5.3 Model 3

In [22]:
model_3_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin"],
    extra_sympy_mappings={ "safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.289
3           4.338e+03  1.665e-02  y = x₂ - 76.746
5           2.056e+00  3.827e+00  y = -2.9767 - (x₅ * x₅)
7           2.022e+00  8.427e-03  y = (x₅ * (0.021657 - x₅)) - 2.9856
9           2.022e+00  4.015e-05  y = (x₅ * ((x₅ * -0.99981) + 0.02139)) - 2.9996
10          2.014e+00  3.987e-03  y = (((sin(x₃) * 0.03415) - x₅) * x₅) - 3.0083
11          2.005e+00  4.276e-03  y = (x₅ * ((x₁ * -0.0017112) - (x₅ + -0.019317))) - 2.9882
12          1.984e+00  1.049e-02  y = (((sin(x₃) * 0.032223) - x₅) * (x₅ - 0.020146)) + -3.0...
                                      148
13          1.926e+00  2.982e-02  y = (x₅ * ((sin(exp(x₄ * 0.70256)) * -0.10427) - x₅)) - 2....
                                      9989
15          1.910e+00  4.216e-03  y = (x₅ * (((sin(exp(x₄ * 0.70289)) * -0.049879) + 0.03364...
    

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                          -76.28898   
	1        0.016651                                     x2 - 76.745514   
	2  >>>>  3.827115                             -2.9766617 - (x5 * x5)   
	3        0.008427              (x5 * (0.021657366 - x5)) - 2.9855542   
	4        0.000040  (x5 * ((x5 * -0.99980634) + 0.021390367)) - 2....   
	5        0.003988  (((sin(x3) * 0.034149744) - x5) * x5) - 3.0082912   
	6        0.004276  (x5 * ((x1 * -0.0017112481) - (x5 + -0.0193169...   
	7        0.010494  (((sin(x3) * 0.032223236) - x5) * (x5 - 0.0201...   
	8        0.029823  (x5 * ((sin(exp(x4 * 0.7025582)) * -0.10427306...   
	9        0.004216  (x5 * (((sin(exp(x4 * 0.7028912)) * -0.0498794...   
	
	          loss  complexity  
	0  4484.837400           1  
	1  4337.941400           3  
	2     2.056337           5  
	3     2.021970           7  
	4     2.021807           9  
	5     2.013761          10  
	6     2.005168          11  
	7     1.984236          12  
	8     1.925933          13  
	9     1.909762          15  
]

In [23]:
best_equation_3 = model_3_wide_range.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 2.9766617
Index 2, score=3.827115
Equation: -2.9766617 - (x5 * x5)

Index 8, score=0.029823
Equation: (x5 * ((sin(exp(x4 * 0.7025582)) * -0.10427306) - x5)) - 2.9989352

Index 1, score=0.016651
Equation: x2 - 76.745514



## 5.5.4 Model 1 with noise std_dev 2 and 5

In [24]:
model_1_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_1_noisy = model_1_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ + -76.488
5           9.261e+00  3.074e+00  y = x₅ * (x₅ * -1.0195)
7           5.635e+00  2.484e-01  y = (x₅ * (x₅ * -0.99853)) + -2.8219
9           5.631e+00  3.751e-04  y = ((x₅ * (x₅ + -0.0077526)) * -0.99872) + -2.8118
11          5.577e+00  4.784e-03  y = ((x₄ * 0.028835) + ((x₅ * x₅) * -0.99859)) + -2.823
12          5.147e+00  8.018e-02  y = ((x₅ * x₅) * -0.9984) + (sin(x₀ * 1.4498) + -2.8634)
14          4.109e+00  1.127e-01  y = (sin((x₀ + x₁) + x₁) + -2.8096) + ((x₅ * x₅) * -0.9988...
                                      5)
16          4.095e+00  1.679e-03  y = sin(x₁ + ((x₀ + x₁) + -0.11085)) + (((x₅ * -0.99877) *...
                                       x₅) + -2.8092)
─────────────────────────────────────────────────────────────────────────────

In [25]:
model_1_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_1_noisy = model_1_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.133
3           4.394e+03  1.726e-02  y = x₂ + -76.59
5           2.874e+01  2.515e+00  y = (x₅ * -1.0227) * x₅
7           2.578e+01  5.427e-02  y = (x₅ * (x₅ * -1.0037)) + -2.5489
9           2.573e+01  9.794e-04  y = ((x₅ + 0.026666) * (x₅ * -1.0031)) + -2.5846
10          2.532e+01  1.615e-02  y = ((x₅ * (x₅ * -1.0035)) + -2.6163) + cos(x₄)
11          2.495e+01  1.475e-02  y = (x₀ * 0.10008) + ((x₅ * (x₅ * -1.003)) + -2.5104)
12          2.467e+01  1.131e-02  y = (((x₅ * x₅) * -1.004) + -2.5131) + sin(x₅ * x₄)
14          2.455e+01  2.485e-03  y = ((x₀ * 0.096406) + (cos(x₄) + (x₅ * (x₅ * -1.0028)))) ...
                                      + -2.5801
15          2.437e+01  7.455e-03  y = ((x₅ * x₅) * -1.0033) + (cos(x₄ + cos(x₁ + x₁)) + -2.5...
                                      949)
16         

[ Info: Final population:
[ Info: Results saved to:


Best by score: x5*(-1.0226563)*x5
Index 2, score=2.514830
Equation: (x5 * -1.0226563) * x5

Index 3, score=0.054270
Equation: (x5 * (x5 * -1.0037082)) + -2.5488727

Index 1, score=0.017257
Equation: x2 + -76.59009



## 5.5.5 Model 2 with noise std_dev 2 and 5

In [26]:
model_2_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_2_noisy = model_2_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ + -76.488
5           5.645e+00  3.322e+00  y = -2.7147 - (x₅ * x₅)
7           5.635e+00  8.622e-04  y = -2.8221 - ((x₅ * 0.99853) * x₅)
9           5.586e+00  4.363e-03  y = ((x₄ * 0.029036) - (x₅ * x₅)) + -2.7199
10          5.159e+00  7.954e-02  y = sin(x₀ * 1.4504) - ((x₅ * x₅) + 2.7464)
13          5.140e+00  1.227e-03  y = -1.4525 - (exp(sin((x₀ * -1.444) + -0.30325)) + (x₅ * ...
                                      x₅))
17          5.125e+00  7.160e-04  y = -1.6351 - ((exp(sin(((x₀ + 0.6586) * -1.4418) + 0.6477...
                                      1)) * 0.8547) + (x₅ * x₅))
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_133219_zFMZti\hall_of_fame.csv


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


Best by score: -x5*x5 - 2.7146783
Index 2, score=3.321948
Equation: -2.7146783 - (x5 * x5)

Index 5, score=0.079541
Equation: sin(x0 * 1.4504242) - ((x5 * x5) + 2.7463548)

Index 1, score=0.016550
Equation: x2 + -76.48811



In [27]:
model_2_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_2_noisy = model_2_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.133
3           4.394e+03  1.726e-02  y = x₂ + -76.589
5           2.585e+01  2.568e+00  y = -2.8207 - (x₅ * x₅)
7           2.578e+01  1.385e-03  y = (x₅ * (-0.03127 - x₅)) + -2.8077
8           2.538e+01  1.561e-02  y = (-2.8703 - (x₅ * x₅)) + cos(x₄)
9           2.516e+01  8.562e-03  y = sin(exp(x₁)) + (-2.916 - (x₅ * x₅))
10          2.474e+01  1.677e-02  y = sin(x₄ * x₅) + (-2.8037 - (x₅ * x₅))
12          2.455e+01  3.913e-03  y = (sin(x₅ * x₄) * 1.6272) - ((x₅ * x₅) + 2.7931)
13          2.448e+01  2.664e-03  y = sin(x₄ * x₅) + ((cos(x₄) + -2.4284) - (x₅ * x₅))
15          2.412e+01  7.436e-03  y = sin(-2.4284 * x₁) + ((-2.4284 - (x₅ * x₅)) + sin(x₅ * ...
                                      x₄))
18          2.401e+01  1.575e-03  y = (((x₂ - x₅) + (-0.04112 - x₂)) * x₅) + (sin((x₁ + x₁) ...
     

## 5.5.6 Model 3 with noise

In [28]:
model_3_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_3_noisy = model_3_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ + -76.483
5           5.645e+00  3.322e+00  y = -2.7147 - (x₅ * x₅)
7           5.635e+00  8.623e-04  y = (x₅ * (x₅ * -0.99853)) + -2.8228
9           5.547e+00  7.834e-03  y = -2.6367 - ((x₅ * x₅) + sin(exp(x₄)))
10          5.159e+00  7.260e-02  y = sin(x₀ * 1.4504) - ((x₅ * x₅) + 2.7463)
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250511_133224_0PYPuc\hall_of_fame.csv
Best by score: -x5*x5 - 2.7146788
Index 2, score=3.321948
Equation: -2.7146788 - (x5 * x5)

Index 5, score=0.072599
Equation: sin(x0 * 1.4504249) - ((x5 * x5) + 2.7463408)

Index 1, score=0.016550
Equation: x2 + -76.48331



In [29]:
model_3_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_3_noisy = model_3_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.133
3           4.394e+03  1.726e-02  y = x₂ + -76.589
5           2.585e+01  2.568e+00  y = -2.8207 - (x₅ * x₅)
7           2.578e+01  1.385e-03  y = ((-0.03127 - x₅) * x₅) + -2.8077
9           2.516e+01  1.209e-02  y = (-2.9158 - (x₅ * x₅)) + sin(exp(x₁))
10          2.474e+01  1.677e-02  y = (-2.8037 - (x₅ * x₅)) + sin(x₅ * x₄)
12          2.430e+01  9.011e-03  y = (-2.8248 - (x₅ * x₅)) + sin(-44.879 - (x₅ * 6.5866))
14          2.420e+01  2.074e-03  y = (-2.8019 - (x₅ * x₅)) + (sin((-25.682 - x₅) * x₅) * 1....
                                      854)
───────────────────────────────────────────────────────────────────────────────────────────────────
Best by score: -x5*x5 - 2.8206582
Index 2, score=2.567903
Equation: -2.8206582 - (x5 * x5)

Index 1, score=0.017257
Equation: x2 + -76.58875

Index 5, 

# 5.6 Output into a DF [4.0]

In [30]:
import pandas as pd

data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5'   
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,x5*x5*(-0.9815461) - 3.1631167,2.031154
1,model_1,3,(x5 * x5) * -1.1931036,2.031154
2,model_1,4,((x5 * x5) * -0.9815461) + -3.1631167,0.560019
3,model_1,9,(cos(cos(x1 + -0.7000452) * x0) + -3.2906103) ...,0.079754
4,model_2,best,(0.42306387**(cos(x0 + x1 + x1 - 1.5642786) - ...,3.142667
5,model_2,3,-3.012776 - (x5 * x5),3.142667
6,model_2,11,"((safe_pow(0.18791056, cos((x1 + -1.5706439) +...",0.806121
7,model_2,13,"(safe_pow(0.42306387, cos(x1 + (x0 + (x1 + -1....",0.724485
8,model_3,best,-x5*x5 - 3.012808,3.201109
9,model_3,3,-3.012808 - (x5 * x5),3.201109


In [31]:
df.to_csv('results4.csv')

In [32]:
data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5'   
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

In [33]:
df.to_csv('results4_only_best.csv')

# 5.7 Function f [5.0]

$$f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$$

- X (random values  $\mathbb{R}^6$ [-5, 5])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev = 0.5) 

In [44]:
import numpy as np
from pysr import PySRRegressor
from math import sin
import sympy

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))


from sympy import primerange
primes_list = [0] + list(primerange(1, 1000))
primes_dict = {i: primes_list[i] if i < len(primes_list) else 0 for i in range(1000)}

y = [
    2.2 * sin(X[i, 0] + 2 * X[i, 1]) - X[i, 5] ** 2 - primes_dict[abs(int(X[i, 0]))]
    for i in range(200)
]

# Dodaj szum
noise = np.random.normal(loc=0.0, scale=0.5, size=200)
y_noisy = np.array(y) + noise

In [45]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.8 Experiments [5.0]



## 5.8.1 Model 1

In [46]:
import sympy
class sympy_p(sympy.Function):
    pass

model_1f = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    timeout_in_seconds=60,
    maxsize=20,
    **default_pysr_params,
)

model_1f.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.176e+01  1.594e+01  y = -11.903
3           6.118e+01  4.750e-03  y = x₂ + -12.056
4           5.765e+01  5.949e-02  y = cos(x₅) + -11.732
5           1.548e+01  1.315e+00  y = x₅ * (x₅ * -1.2424)
7           8.495e+00  3.000e-01  y = ((x₅ * -0.98039) * x₅) + -3.9175
9           8.481e+00  8.454e-04  y = (x₅ * ((x₅ + 0.043716) * -0.97735)) + -3.9362
10          7.420e+00  1.335e-01  y = (cos(x₀) + ((x₅ * x₅) * -0.98748)) + -3.592
11          2.735e+00  9.980e-01  y = ((x₅ * x₅) * -1.0023) + (x₀ * (x₀ * -0.36961))
13          2.450e+00  5.509e-02  y = (x₅ * (x₅ * -0.96323)) + ((x₀ * (x₀ * -0.32552)) + -1....
                                      0127)
14          2.367e+00  3.455e-02  y = (((x₅ * x₅) * -0.96726) + -5.5221) + (cos(x₀ * -0.4369...
                                      2) * 4.9398)
16          2.352e+00  3.221e-03  y =

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.903449   
	1         0.004750                                     x2 + -12.05577   
	2         0.059494                               cos(x5) + -11.732496   
	3         1.314799                             x5 * (x5 * -1.2424145)   
	4         0.300026             ((x5 * -0.98039085) * x5) + -3.9175193   
	5         0.000846  (x5 * ((x5 + 0.043716386) * -0.9773465)) + -3....   
	6         0.133550  (cos(x0) + ((x5 * x5) * -0.98747545)) + -3.591...   
	7   >>>>  0.997968  ((x5 * x5) * -1.0023342) + (x0 * (x0 * -0.3696...   
	8         0.055092  (x5 * (x5 * -0.96322936)) + ((x0 * (x0 * -0.32...   
	9         0.034551  (((x5 * x5) * -0.96725845) + -5.5221243) + (co...   
	10        0.003221  ((cos(x0 * 0.43638465) * 4.9500933) + (x5 * ((...   
	11        0.016386  ((x5 * -0.96241075) * x5) + ((cos(x0 * -0.4470...   
	12        0.001029  (sin(x0) * -0.420477) + ((cos(x0 * -0.42758664...   
	13        0.000461  ((((x5 * -0.96281433) * x5) + (sin(sin(x0)) * ...   
	
	         loss  complexity  
	0   61.764057           1  
	1   61.180126           3  
	2   57.646430           4  
	3   15.479691           5  
	4    8.494999           7  
	5    8.480646           9  
	6    7.420428          10  
	7    2.735377          11  
	8    2.449992          13  
	9    2.366788          14  
	10   2.351591          16  
	11   2.275775          18  
	12   2.273436          19  
	13   2.272389          20  
]

### Model 1 best + top3 according to score

In [47]:
best_equation_1 = model_1f.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1f.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x0*x0*(-0.36960706) + x5*x5*(-1.0023342)
Index 3, score=1.314799
Equation: x5 * (x5 * -1.2424145)

Index 7, score=0.997968
Equation: ((x5 * x5) * -1.0023342) + (x0 * (x0 * -0.36960706))

Index 4, score=0.300026
Equation: ((x5 * -0.98039085) * x5) + -3.9175193



## 5.8.2 Model 2

Added constraint to operator "^" - safe_pow

In [ ]:
def safe_pow(x, y):
    if x < 0 and np.floor(y) != y:
        return np.nan
    return np.power(x, y)

model_2f = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.159
3           5.537e+01  2.051e-03  y = -11.021 - x₅
4           5.152e+01  7.208e-02  y = cos(x₅) - 10.987
5           2.224e+00  3.143e+00  y = -3.0129 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = (x₅ * (x₅ * -0.98154)) - 3.1631
9           2.194e+00  2.514e-03  y = (-2.9964 - (x₀ * -0.05681)) - (x₅ * x₅)
11          2.132e+00  1.432e-02  y = ((x₅ + (x₄ * -0.035437)) * (x₅ * -0.97745)) - 3.1981
13          2.120e+00  2.745e-03  y = ((((x₄ * -0.035437) + x₅) * (x₅ + 0.038744)) * -0.9774...
                                      5) - 3.1981
14          6.613e-01  1.165e+00  y = (-3.0203 - cos(x₀ - ((x₁ - -0.79503) * -2.003))) - (x₅...
                                       * x₅)
16          6.599e-01  1.023e-03  y = (-3.0105 - cos(x₀ - ((x₁ - -0.79435) * -2.002))) - (x₅...
                         

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.158516   
	1         0.002051                                    -11.020784 - x5   
	2         0.072084                                cos(x5) - 10.987488   
	3         3.142667                             -3.0128691 - (x5 * x5)   
	4         0.004262                 (x5 * (x5 * -0.981545)) - 3.163129   
	5         0.002514      (-2.9964285 - (x0 * -0.05681007)) - (x5 * x5)   
	6         0.014322  ((x5 + (x4 * -0.03543666)) * (x5 * -0.9774493)...   
	7         0.002745  ((((x4 * -0.03543666) + x5) * (x5 + 0.03874399...   
	8         1.165197  (-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2...   
	9         0.001023  (-3.01047 - cos(x0 - ((x1 - -0.79435456) * -2....   
	10        0.002642  (-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2...   
	11        0.004064  (-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2...   
	12        0.004143  ((-3.01047 - cos(x0 - ((x1 - -0.79435456) * -2...   
	13        0.004617  (-3.01047 - cos(x0 - ((x1 - -0.79435456) * -2....   
	14        0.006367  (-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2...   
	15  >>>>  3.442063  ((-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -...   
	
	         loss  complexity  
	0   55.599796           1  
	1   55.372204           3  
	2   51.521240           4  
	3    2.224043           5  
	4    2.205166           7  
	5    2.194107           9  
	6    2.132152          11  
	7    2.120477          13  
	8    0.661295          14  
	9    0.659943          16  
	10   0.656465          18  
	11   0.653803          19  
	12   0.651100          20  
	13   0.645115          22  
	14   0.628893          26  
	15   0.020124          27  
]

  - outputs\20250510_133001_UtWRwD\hall_of_fame.csv


In [ ]:
best_equation_2 = model_2f.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2f.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -(x5*x5 - (-1)*(-0.7950349)*0.061814636) - cos(x0 - (-2.0029647)*(x1 - 1*(-0.7950349))) - cos(x0 - (-2.0029647)*(x1 - 1*(-0.7950349))) - 3.0203273
Index 15, score=3.442063
Equation: ((-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2.0029647))) - cos(x0 - ((x1 - -0.7950349) * -2.0029647))) - ((x5 * x5) - (-0.061814636 * -0.7950349))

Index 3, score=3.142667
Equation: -3.0128691 - (x5 * x5)

Index 8, score=1.165197
Equation: (-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2.0029647))) - (x5 * x5)



## 5.8.3 Model 3

In [ ]:
model_3f = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_noisy = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  1.594e+01  y = -11.158
3           5.537e+01  2.051e-03  y = -11.021 - x₅
4           5.462e+01  1.364e-02  y = -11.157 - sin(x₀)
5           2.224e+00  3.201e+00  y = -3.0128 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = -3.1631 - ((x₅ * 0.98155) * x₅)
9           2.153e+00  1.189e-02  y = (((x₄ * 0.03329) - x₅) * x₅) - 3.0191
10          2.134e+00  9.020e-03  y = (-3.0123 - (x₅ * x₅)) + (sin(x₀) * -0.41058)
11          2.133e+00  4.447e-04  y = (-3.0114 - (x₅ * x₅)) + (sin(sin(x₀)) * -0.47021)
12          2.106e+00  1.284e-02  y = (((sin(x₀) * -0.44337) + -3.2709) - (x₅ * x₅)) * 0.977...
                                      35
13          2.099e+00  3.037e-03  y = (-3.0123 - (x₅ * x₅)) + sin(sin(x₀ * (0.81933 + x₁)))
14          1.707e+00  2.068e-01  y = -2.9841 - (((sin(x₁) * sin(x₁)) * sin(x₀)) + (x₅ * x₅)...
    

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -11.158203   
	1         0.002051                                    -11.020794 - x5   
	2         0.013642                               -11.157449 - sin(x0)   
	3   >>>>  3.201109                             -3.0128005 - (x5 * x5)   
	4         0.004262              -3.1631155 - ((x5 * 0.98154515) * x5)   
	5         0.011890       (((x4 * 0.033289738) - x5) * x5) - 3.0191004   
	6         0.009021  (-3.0123389 - (x5 * x5)) + (sin(x0) * -0.41058...   
	7         0.000445  (-3.01142 - (x5 * x5)) + (sin(sin(x0)) * -0.47...   
	8         0.012839  (((sin(x0) * -0.44336697) + -3.2709334) - (x5 ...   
	9         0.003038  (-3.0123389 - (x5 * x5)) + sin(sin(x0 * (0.819...   
	10        0.206831  -2.984116 - (((sin(x1) * sin(x1)) * sin(x0)) +...   
	
	         loss  complexity  
	0   55.599790           1  
	1   55.372204           3  
	2   54.621952           4  
	3    2.224043           5  
	4    2.205166           7  
	5    2.153344           9  
	6    2.134007          10  
	7    2.133058          11  
	8    2.105847          12  
	9    2.099460          13  
	10   1.707190          14  
]

In [ ]:
best_equation_3 = model_3f.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3f.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 3.0128005
Index 3, score=3.201109
Equation: -3.0128005 - (x5 * x5)

Index 10, score=0.206831
Equation: -2.984116 - (((sin(x1) * sin(x1)) * sin(x0)) + (x5 * x5))

Index 2, score=0.013642
Equation: -11.157449 - sin(x0)



## 5.8.4 Model 1 with noise

In [ ]:
model_1f_noisy.fit(X, y_noisy)
best_equation_1_noisy = model_1f_noisy.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1f_noisy.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.094
3           5.560e+01  5.027e-04  y = x₂ + -11.245
4           5.159e+01  7.493e-02  y = cos(x₅) + -10.922
5           6.855e+00  2.018e+00  y = x₅ * (x₅ * -1.188)
7           2.449e+00  5.148e-01  y = ((x₅ * x₅) * -0.97982) + -3.1117
9           2.434e+00  2.937e-03  y = (((x₅ * -0.97674) + -0.04277) * x₅) + -3.1309
11          2.415e+00  3.898e-03  y = ((x₀ * 0.059929) + (x₅ * (x₅ * -0.9786))) + -3.1043
12          2.340e+00  3.182e-02  y = ((x₅ * (x₅ * -0.98091)) + sin(x₁ * -2.0207)) + -3.1691
14          2.339e+00  1.214e-04  y = (sin(x₁ * -2.0207) + (x₅ * ((x₅ * -0.98031) + -0.00851...
                                      56))) + -3.1729
16          2.318e+00  4.442e-03  y = ((x₀ * 0.047973) + ((x₅ * (x₅ * -0.97998)) + sin(x₁ * ...
                                      -2.0232))) + -3.1624
17  

## 5.8.5 Model 2 with noise

In [ ]:
model_2f_noisy.fit(X, y_noisy)
best_equation_2_noisy = model_2f_noisy.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2f_noisy.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.092
3           5.528e+01  3.374e-03  y = -10.955 - x₅
4           5.159e+01  6.919e-02  y = cos(x₅) + -10.922
5           2.471e+00  3.039e+00  y = -2.9473 - (x₅ * x₅)
7           2.449e+00  4.586e-03  y = -3.1116 - ((x₅ * x₅) * 0.97983)
9           2.434e+00  2.937e-03  y = -3.1309 - (((x₅ + 0.043784) * x₅) * 0.97674)
12          2.347e+00  1.218e-02  y = (((x₅ * (x₅ + x₅)) + sin(x₀)) * -0.48744) - 3.1514
13          2.345e+00  6.197e-04  y = (((x₅ * (x₅ + x₅)) + sin(sin(x₀))) - -6.4105) * -0.488...
                                      49
16          2.332e+00  1.956e-03  y = -1.6727 - (((-3.0488 - ((x₅ + x₅) * x₅)) + cos(x₅ * -6...
                                      .0988)) * -0.48545)
18          2.322e+00  1.989e-03  y = -3.1526 - ((((0.33804 - ((x₅ + x₅) + 0.40742)) * x₅) +...
                 

## 5.8.6 Model 3 with noise

In [ ]:
model_3f_noisy.fit(X, y_noisy)
best_equation_3_noisy = model_3f_noisy.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3f_noisy.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  1.594e+01  y = -11.093
3           5.528e+01  3.374e-03  y = -10.955 - x₅
4           5.467e+01  1.109e-02  y = -11.092 - sin(x₀)
5           2.471e+00  3.097e+00  y = -2.9473 - (x₅ * x₅)
7           2.449e+00  4.586e-03  y = (x₅ * (x₅ * -0.97982)) - 3.1117
9           2.414e+00  7.170e-03  y = -3.1116 - ((x₅ + (x₄ * -0.03746)) * x₅)
11          2.363e+00  1.062e-02  y = (((x₅ + (x₄ * -0.03746)) * x₅) * -0.97839) + -3.1303
12          2.363e+00  1.019e-05  y = ((x₅ + sin(x₄ * -0.037625)) * (x₅ * -0.9784)) + -3.130...
                                      3
13          2.344e+00  7.932e-03  y = (((x₄ * -0.03855) + x₅) * ((x₅ * -0.97483) - 0.048912)...
                                      ) + -3.1528
14          2.344e+00  9.656e-06  y = ((x₅ + sin(x₄ * -0.038725)) * ((x₅ * -0.97484) - 0.048...
                             

# 5.9 Output into a DF [5.0: 3.0]

In [ ]:
import pandas as pd

data = []

for model_name in ['model_1f', 'model_2f', 'model_3f','model_1f_noisy', 'model_2f_noisy', 'model_3f_noisy']:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,x5*x5*(-0.98154604) - 3.163106,2.031154
1,model_1,3,(x5 * x5) * -1.1931325,2.031154
2,model_1,4,(x5 * (x5 * -0.98154604)) + -3.163106,0.560019
3,model_1,2,cos(x5) + -10.987488,0.073704
4,model_2,best,-(x5*x5 - (-1)*(-0.7950349)*0.061814636) - cos...,3.442063
5,model_2,15,((-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -...,3.442063
6,model_2,3,-3.0128691 - (x5 * x5),3.142667
7,model_2,8,(-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2...,1.165197
8,model_3,best,-x5*x5 - 3.0128005,3.201109
9,model_3,3,-3.0128005 - (x5 * x5),3.201109


In [ ]:
df.to_csv('results5_for3.csv')

# 5.10 Function f [5.0: 4.0]

$$f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$$

- X (random values  $\mathbb{R}^6$ [-15, 15])
- y (values without noise) 
- y_noisy (values with noise with mean = 0.0 and std_dev1 = 2, std_dev2 = 5) 

In [ ]:
from numpy import sin, cos, exp, log, sqrt
# Dataset

np.random.seed(0)
X = np.random.uniform(-15, 15, size=(200, 6))
# y = 2.5382 * np.cos(X[:, 3]) + X[:, 0] ** 2 - 2
from sympy import primerange
primes_list = [0] + list(primerange(1, 1000))
primes_dict = {i: primes_list[i] if i < len(primes_list) else 0 for i in range(1000)}

y = [
    2.2 * sin(X[i, 0] + 2 * X[i, 1]) - X[i, 5] ** 2 - primes_dict[abs(int(X[i, 0]))]
    for i in range(200)
]
# Add the y samples with random noise
noise1 = np.random.normal(loc=0.0, scale=2, size=len(y))
y_noisy1 = np.array(y) + noise1
noise2 = np.random.normal(loc=0.0, scale=5, size=len(y))
y_noisy2 = np.array(y) + noise2

In [ ]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

# 5.11 Experiments [5.0]



## 5.11.1 Model 1

In [ ]:
model_1f_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    extra_sympy_mappings={"p": sympy_p},
    maxsize=20,
    **default_pysr_params,
)

model_1f_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.289
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           6.228e+00  3.273e+00  y = x₅ * (x₅ * -1.0218)
7           2.054e+00  5.546e-01  y = (x₅ * (x₅ * -0.99929)) + -3.0285
9           2.022e+00  7.920e-03  y = ((x₅ + -0.021392) * (x₅ * -0.99981)) + -2.9996
12          1.997e+00  4.119e-03  y = ((x₅ * (x₅ + (cos(x₃) * -0.040072))) * -0.99963) + -3....
                                      0105
13          1.959e+00  1.925e-02  y = ((x₅ + (cos(exp(x₄)) * -0.043225)) * (x₅ * -0.99989)) ...
                                      + -2.9933
14          1.941e+00  9.093e-03  y = (((x₅ * -0.99945) * x₅) + -3.6752) + sin(exp(sin(x₂ * ...
                                      4.2217)))
16          1.923e+00  4.800e-03  y = (sin(sin(exp(x₅ * -1.0461) * -10.061)) + -2.9474) + ((...
                       

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          -76.28898   
	1         0.016651                                    x2 + -76.745125   
	2         3.273026                             x5 * (x5 * -1.0218029)   
	3   >>>>  0.554636               (x5 * (x5 * -0.9992922)) + -3.028511   
	4         0.007920  ((x5 + -0.02139206) * (x5 * -0.99980634)) + -2...   
	5         0.004119  ((x5 * (x5 + (cos(x3) * -0.040072292))) * -0.9...   
	6         0.019247  ((x5 + (cos(exp(x4)) * -0.043224953)) * (x5 * ...   
	7         0.009093  (((x5 * -0.99944854) * x5) + -3.6751559) + sin...   
	8         0.004800  (sin(sin(exp(x5 * -1.0460858) * -10.0606)) + -...   
	9         0.000046  ((x5 * x5) * -1.0004921) + (sin(sin(sin(exp(x5...   
	10        0.003416  (sin(sin((exp(x5 * -1.0574919) * -10.058729) +...   
	11        0.005275  (sin(sin(sin((exp(x5 * -1.0574919) * -10.05872...   
	12        0.012235  -2.9428918 + (((x5 * x5) * -1.0004921) + sin(s...   
	
	           loss  complexity  
	0   4484.837400           1  
	1   4337.941400           3  
	2      6.228301           5  
	3      2.054086           7  
	4      2.021807           9  
	5      1.996978          12  
	6      1.958909          13  
	7      1.941177          14  
	8      1.922632          16  
	9      1.922543          17  
	10     1.915988          18  
	11     1.905907          19  
	12     1.882730          20  
]

### Model 1 best + top3 according to score

In [ ]:
best_equation_1 = model_1f_wide_range.sympy()
print("Best by score:", best_equation_1)


top3_1 = model_1f_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_1.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: x5*x5*(-0.9992922) - 3.028511
Index 2, score=3.273026
Equation: x5 * (x5 * -1.0218029)

Index 3, score=0.554636
Equation: (x5 * (x5 * -0.9992922)) + -3.028511

Index 6, score=0.019247
Equation: ((x5 + (cos(exp(x4)) * -0.043224953)) * (x5 * -0.99989194)) + -2.9932628



## 5.11.2 Model 2

Added constraint to operator "^" - safe_pow

In [ ]:
def safe_pow(x, y):
    if x < 0 and np.floor(y) != y:
        return np.nan
    return np.power(x, y)

model_2f_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["cos", "exp", "sin", "log"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=30,
    **default_pysr_params,
)

model_2f_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.289
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           2.056e+00  3.827e+00  y = -2.9766 - (x₅ * x₅)
7           2.022e+00  8.427e-03  y = (x₅ * (0.021677 - x₅)) - 2.9856
9           2.022e+00  4.003e-05  y = -2.9995 - ((x₅ * (0.021389 - x₅)) * -0.99981)
10          2.017e+00  2.292e-03  y = -2.9974 - ((x₅ - (sin(x₄) * -0.031745)) * x₅)
11          2.016e+00  4.257e-04  y = -2.9974 - (x₅ * (x₅ - (sin(sin(x₄)) * -0.031745)))
12          1.984e+00  1.603e-02  y = (x₅ * (0.02013 - (x₅ - (sin(x₃) * 0.032212)))) - 3.014...
                                      8
13          1.980e+00  2.295e-03  y = -1.9439 - (safe_pow(1.4762, sin(exp(x₅))) - (x₅ * (0.0...
                                      23913 - x₅)))
14          1.913e+00  3.438e-02  y = ((x₅ * (0.013511 - x₅)) - (sin(x₂ + x₃) * 0.50496)) -

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         -76.288925   
	1         0.016651                                     x2 + -76.74522   
	2   >>>>  3.827115                             -2.9765701 - (x5 * x5)   
	3         0.008427               (x5 * (0.02167654 - x5)) - 2.9855618   
	4         0.000040  -2.9995186 - ((x5 * (0.021388775 - x5)) * -0.9...   
	5         0.002292  -2.9973736 - ((x5 - (sin(x4) * -0.03174465)) *...   
	6         0.000426  -2.9973736 - (x5 * (x5 - (sin(sin(x4)) * -0.03...   
	7         0.016027  (x5 * (0.02012969 - (x5 - (sin(x3) * 0.0322119...   
	8         0.002295  -1.9439019 - (safe_pow(1.4761515, sin(exp(x5))...   
	9         0.034384  ((x5 * (0.013510874 - x5)) - (sin(x2 + x3) * 0...   
	10        0.005004  (((0.013940909 - x5) - (sin((x3 - sin(x1)) + 1...   
	11        0.000163  (((0.013940909 - x5) - (sin(1.0355971 + (x3 - ...   
	12        0.010863  (((0.015797224 - x5) - (sin(sin(cos(x5 * -1.86...   
	
	           loss  complexity  
	0   4484.837400           1  
	1   4337.941400           3  
	2      2.056337           5  
	3      2.021969           7  
	4      2.021807           9  
	5      2.017178          10  
	6      2.016319          11  
	7      1.984261          12  
	8      1.979712          13  
	9      1.912798          14  
	10     1.884299          17  
	11     1.883684          19  
	12     1.803589          23  
]

In [ ]:
best_equation_2 = model_2f_wide_range.sympy()
print("Best by score:", best_equation_2)


top3_2 = model_2f_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_2.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 2.9765701
Index 2, score=3.827115
Equation: -2.9765701 - (x5 * x5)

Index 9, score=0.034384
Equation: ((x5 * (0.013510874 - x5)) - (sin(x2 + x3) * 0.5049631)) - 3.0153282

Index 1, score=0.016651
Equation: x2 + -76.74522



## 5.11.3 Model 3

In [ ]:
model_3f_wide_range = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y)},
    maxsize=15,
    **default_pysr_params,
)

model_3f_noisy_std_dev_2 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_noisy_std_dev_5 = PySRRegressor(
    niterations=30,
    binary_operators=["+", "*", "-", "safe_pow(x, y) = x^y"],
    unary_operators=["exp", "sin", "p"],
    extra_sympy_mappings={"safe_pow": lambda x, y: sympy.Pow(x, y), "p": sympy_p},
    maxsize=15,
    **default_pysr_params,
)

model_3f_wide_range.fit(X, y)

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.485e+03  1.594e+01  y = -76.292
3           4.338e+03  1.665e-02  y = x₂ + -76.745
5           2.056e+00  3.827e+00  y = -2.977 - (x₅ * x₅)
7           2.022e+00  8.427e-03  y = ((0.021677 - x₅) * x₅) - 2.9856
9           2.022e+00  4.026e-05  y = (x₅ * (0.021394 - (x₅ * 0.99981))) - 2.9996
11          2.005e+00  4.133e-03  y = ((x₅ - (x₁ * -0.0017115)) * (0.019323 - x₅)) + -2.9882
12          6.141e-01  1.183e+00  y = -3.0351 - ((x₅ * x₅) - sin(x₀ + (x₁ + x₁)))
14          6.118e-01  1.871e-03  y = -2.9663 - (((x₅ * x₅) - sin((x₁ + x₁) + x₀)) + 0.02094...
                                      1)
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250510_133017_ljCZ5A\hall_of_fame.csv


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                          -76.29248   
	1        0.016651                                    x2 + -76.745186   
	2        3.827115                             -2.9769638 - (x5 * x5)   
	3        0.008427               ((0.021677315 - x5) * x5) - 2.985569   
	4        0.000040  (x5 * (0.021393938 - (x5 * 0.99980634))) - 2.9...   
	5        0.004133  ((x5 - (x1 * -0.0017115241)) * (0.019323384 - ...   
	6  >>>>  1.183330     -3.0351326 - ((x5 * x5) - sin(x0 + (x1 + x1)))   
	7        0.001871  -2.9662995 - (((x5 * x5) - sin((x1 + x1) + x0)...   
	
	          loss  complexity  
	0  4484.837400           1  
	1  4337.941400           3  
	2     2.056337           5  
	3     2.021970           7  
	4     2.021807           9  
	5     2.005164          11  
	6     0.614096          12  
	7     0.611802          14  
]

In [ ]:
best_equation_3 = model_3f_wide_range.sympy()
print("Best by score:", best_equation_3)


top3_3 = model_3f_wide_range.equations_.nlargest(3, "score")

for idx, row in top3_3.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

Best by score: -x5*x5 - 3.0128005
Index 2, score=3.827115
Equation: -2.9769638 - (x5 * x5)

Index 6, score=1.183330
Equation: -3.0351326 - ((x5 * x5) - sin(x0 + (x1 + x1)))

Index 1, score=0.016651
Equation: x2 + -76.745186



## 5.11.4 Model 1 with noise std_dev 2 and 5

In [ ]:
model_1f_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_1_noisy = model_1f_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1f_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()


c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ + -76.487
5           9.261e+00  3.074e+00  y = x₅ * (x₅ * -1.0195)
7           5.635e+00  2.484e-01  y = ((x₅ * -0.99853) * x₅) + -2.8224
9           5.631e+00  3.751e-04  y = (x₅ * ((x₅ * -0.99871) + 0.0077253)) + -2.8127
11          5.541e+00  8.079e-03  y = (((x₅ * x₅) + sin(exp(x₄))) * -0.99876) + -2.7276
12          5.147e+00  7.359e-02  y = (((x₅ * x₅) + sin(x₀ * -1.4498)) * -0.9984) + -2.8636
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250510_133018_aNl7st\hall_of_fame.csv


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


Best by score: x5*(-0.9985297)*x5 - 2.822441
Index 2, score=3.074411
Equation: x5 * (x5 * -1.0195066)

Index 3, score=0.248400
Equation: ((x5 * -0.9985297) * x5) + -2.822441

Index 6, score=0.073594
Equation: (((x5 * x5) + sin(x0 * -1.4498099)) * -0.9983978) + -2.8635705



In [ ]:
model_1f_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_1_noisy = model_1f_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_1_noisy)


top3_1_noisy = model_1f_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_1_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.127
3           4.394e+03  1.726e-02  y = x₂ + -76.589
5           2.874e+01  2.515e+00  y = (x₅ * -1.0226) * x₅
7           2.578e+01  5.427e-02  y = ((x₅ * -1.0037) * x₅) + -2.5487
9           2.573e+01  9.792e-04  y = ((x₅ + 0.026661) * (x₅ * -1.0031)) + -2.5849
10          2.532e+01  1.615e-02  y = ((x₅ * (x₅ * -1.0035)) + -2.6162) + cos(x₄)
11          2.495e+01  1.475e-02  y = ((x₅ * (x₅ * -1.003)) + (x₀ * 0.10008)) + -2.511
12          2.467e+01  1.130e-02  y = sin(x₄ * x₅) + ((x₅ * (x₅ * -1.0041)) + -2.5119)
14          2.455e+01  2.487e-03  y = (x₀ * 0.096418) + ((((x₅ * -1.0028) * x₅) + cos(x₄)) +...
                                       -2.58)
15          2.445e+01  4.081e-03  y = (((x₅ * -1.0042) * x₅) + sin(sin(x₅ * x₄) * 2.1258)) +...
                                       -2.4642
16      

[ Info: Final population:
[ Info: Results saved to:


Best by score: x5*(-1.0226468)*x5
Index 2, score=2.514830
Equation: (x5 * -1.0226468) * x5

Index 3, score=0.054270
Equation: ((x5 * -1.0037091) * x5) + -2.5486984

Index 10, score=0.020387
Equation: (x0 * 0.0964179) + ((x5 * (x5 * -1.002817)) + (sin(x5 * x4) + -2.5799537))

  - outputs\20250510_133020_mmY2tz\hall_of_fame.csv


## 5.11.5 Model 2 with noise std_dev 2 and 5

In [ ]:
model_2f_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_2_noisy = model_2f_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2f_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.027
3           4.336e+03  1.655e-02  y = x₂ + -76.483
5           5.645e+00  3.322e+00  y = -2.7146 - (x₅ * x₅)
7           5.636e+00  7.914e-04  y = (-2.8636 - (x₅ * x₅)) * 0.9984
9           5.558e+00  6.951e-03  y = -2.7397 - ((x₅ * x₅) + sin(exp(x₄)))
10          5.358e+00  3.671e-02  y = -2.7737 - (sin(x₅ * 6.6668) + (x₅ * x₅))
11          5.341e+00  3.061e-03  y = -2.7737 - (sin(sin(x₅ * 6.6668)) + (x₅ * x₅))
12          5.147e+00  3.697e-02  y = -2.8636 - ((sin(x₀ * -1.4498) + (x₅ * x₅)) * 0.9984)
15          5.062e+00  5.569e-03  y = -2.7941 - (cos((cos(x₁ + -0.72545) * 2.5197) + x₀) + (...
                                      x₅ * x₅))
16          4.787e+00  5.592e-02  y = -2.7453 - (cos((x₀ + ((x₁ * 1.9137) + 1.3506)) + 0.280...
                                      81) + (x₅ * x₅))
21       

In [ ]:
model_2f_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_2_noisy = model_2f_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_2_noisy)


top3_2_noisy = model_2f_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_2_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.134
3           4.394e+03  1.726e-02  y = x₂ + -76.59
5           2.585e+01  2.568e+00  y = -2.8206 - (x₅ * x₅)
7           2.578e+01  1.385e-03  y = -2.8077 - (x₅ * (x₅ - -0.03127))
8           2.538e+01  1.561e-02  y = cos(x₄) - ((x₅ * x₅) - -2.8703)
9           2.499e+01  1.522e-02  y = (x₀ * 0.10118) + (-2.7328 - (x₅ * x₅))
10          2.474e+01  1.011e-02  y = sin(x₄ * x₅) - ((x₅ * x₅) - -2.8036)
12          2.445e+01  5.852e-03  y = (-3.0897 - sin((x₅ * x₁) * -4.2154)) - (x₅ * x₅)
13          2.430e+01  6.192e-03  y = sin(x₅ * x₄) + ((cos(x₄) - (x₅ * x₅)) + -2.8535)
14          2.391e+01  1.646e-02  y = (-2.5677 - (x₅ * x₅)) - (sin((x₅ * x₁) * -4.2178) * 1....
                                      9866)
15          2.373e+01  7.347e-03  y = (-2.8513 - (sin(sin((x₁ * x₅) * -4.2165)) * 2.2888)) -...


## 5.11.6 Model 3 with noise

In [ ]:
model_3f_noisy_std_dev_2.fit(X, y_noisy1)
best_equation_3_noisy = model_3f_noisy_std_dev_2.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3f_noisy_std_dev_2.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  1.594e+01  y = -76.022
3           4.336e+03  1.655e-02  y = x₂ + -76.484
5           5.645e+00  3.322e+00  y = -2.7146 - (x₅ * x₅)
7           5.635e+00  8.625e-04  y = (x₅ * (x₅ * -0.99853)) - 2.8226
9           5.631e+00  3.752e-04  y = ((x₅ * (x₅ + -0.0077526)) * -0.99871) - 2.8123
11          5.541e+00  8.060e-03  y = ((x₅ * (x₅ * -0.99878)) + -2.7268) - sin(exp(x₄))
12          5.228e+00  5.816e-02  y = ((sin(1.4902 * x₀) - (x₅ * x₅)) - 2.9947) + 0.25353
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs\20250510_133027_1Dah7x\hall_of_fame.csv
Best by score: -x5*x5 - 2.7146482
Index 2, score=3.321948
Equation: -2.7146482 - (x5 * x5)

Index 6, score=0.058159
Equation: ((sin(1.490223 * x0) - (x5 * x5)) - 2.9946597) + 0.2535261

Index 1, score=0.016550
Equatio

In [ ]:
model_3f_noisy_std_dev_5.fit(X, y_noisy2)
best_equation_3_noisy = model_3f_noisy_std_dev_5.sympy()
print("Best by score:", best_equation_3_noisy)


top3_3_noisy = model_3f_noisy_std_dev_5.equations_.nlargest(3, "score")

for idx, row in top3_3_noisy.iterrows():
    print(f"Index {idx}, score={row.score:.6f}")
    print("Equation:", row.equation)
    print()

c:\Users\Adam\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysr\sr.py:2776: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.548e+03  1.594e+01  y = -76.133
3           4.394e+03  1.726e-02  y = x₂ + -76.589
5           2.585e+01  2.568e+00  y = -2.8206 - (x₅ * x₅)
7           2.578e+01  1.385e-03  y = (x₅ * (-0.03127 - x₅)) + -2.8077
9           2.499e+01  1.541e-02  y = (x₀ * 0.10117) + (-2.7326 - (x₅ * x₅))
10          2.474e+01  1.011e-02  y = sin(x₄ * x₅) + (-2.8036 - (x₅ * x₅))
12          2.467e+01  1.476e-03  y = ((sin(x₄ * x₅) - (x₅ * x₅)) * 1.004) - 2.5082
14          2.449e+01  3.564e-03  y = (sin(x₄ * x₅) + ((x₁ * 0.054889) + -2.7712)) - (x₅ * x...
                                      ₅)
15          2.387e+01  2.574e-02  y = (sin(x₁ * 3.2167) + (sin(x₅ * x₄) + -2.7305)) - (x₅ * ...
                                      x₅)
───────────────────────────────────────────────────────────────────────────────────────────────────


[ Info: Final population:
[ Info: Results saved to:


Best by score: -x5*x5 - 2.820567
Index 2, score=2.567903
Equation: -2.820567 - (x5 * x5)

Index 8, score=0.025742
Equation: (sin(x1 * 3.216724) + (sin(x5 * x4) + -2.730506)) - (x5 * x5)

Index 1, score=0.017257
Equation: x2 + -76.58948

  - outputs\20250510_133029_NeDaRF\hall_of_fame.csv


# 5.12 Output into a DF [5.0]

In [ ]:
import pandas as pd

data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5',
    'model_1f', 'model_2f', 'model_3f',
    'model_1f_noisy', 'model_2f_noisy', 'model_3f_noisy',
    'model_1f_wide_range', 'model_2f_wide_range', 'model_3f_wide_range',
    'model_1f_noisy_std_dev_2', 'model_2f_noisy_std_dev_2', 'model_3f_noisy_std_dev_2',
    'model_1f_noisy_std_dev_5', 'model_2f_noisy_std_dev_5', 'model_3f_noisy_std_dev_5',    
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

    # Add data for the top 3 equations
    for idx, row in top3.iterrows():
        data.append([model_name, idx, row['equation'], row['score']])

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

display(df)

,Model,Equation_Type,Equation,Score
0,model_1,best,x5*x5*(-0.98154604) - 3.163106,2.031154
1,model_1,3,(x5 * x5) * -1.1931325,2.031154
2,model_1,4,(x5 * (x5 * -0.98154604)) + -3.163106,0.560019
3,model_1,2,cos(x5) + -10.987488,0.073704
4,model_2,best,-(x5*x5 - (-1)*(-0.7950349)*0.061814636) - cos...,3.442063
5,model_2,15,((-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -...,3.442063
6,model_2,3,-3.0128691 - (x5 * x5),3.142667
7,model_2,8,(-3.0203273 - cos(x0 - ((x1 - -0.7950349) * -2...,1.165197
8,model_3,best,-x5*x5 - 3.0128005,3.201109
9,model_3,3,-3.0128005 - (x5 * x5),3.201109


In [ ]:
df.to_csv('results5.csv')

In [ ]:
data = []
models = [
    'model_1', 'model_2', 'model_3',
    'model_1_noisy', 'model_2_noisy', 'model_3_noisy',
    'model_1_wide_range', 'model_2_wide_range', 'model_3_wide_range',
    'model_1_noisy_std_dev_2', 'model_2_noisy_std_dev_2', 'model_3_noisy_std_dev_2',
    'model_1_noisy_std_dev_5', 'model_2_noisy_std_dev_5', 'model_3_noisy_std_dev_5',
    'model_1f', 'model_2f', 'model_3f',
    'model_1f_noisy', 'model_2f_noisy', 'model_3f_noisy',
    'model_1f_wide_range', 'model_2f_wide_range', 'model_3f_wide_range',
    'model_1f_noisy_std_dev_2', 'model_2f_noisy_std_dev_2', 'model_3f_noisy_std_dev_2',
    'model_1f_noisy_std_dev_5', 'model_2f_noisy_std_dev_5', 'model_3f_noisy_std_dev_5',    
]

for model_name in models:
    model = globals()[model_name]  # Get the model object from the global namespace

    best_equation = model.sympy()
    top3 = model.equations_.nlargest(3, "score")

    # Add data for the best equation
    data.append([model_name, 'best', best_equation, top3.iloc[0]['score']])  # Changed to iloc to access row

df = pd.DataFrame(data, columns=['Model', 'Equation_Type', 'Equation', 'Score'])

In [ ]:
df.to_csv('results5_only_best.csv')